# HealthTrack Great Expectations Checkpoint

In [ ]:
import shutil
from pathlib import Path

import great_expectations as gx
import pandas as pd

df = pd.read_csv('healthtrack_raw.csv')

context = gx.get_context(mode='ephemeral')
datasource = context.data_sources.add_pandas(name='healthtrack_pandas')
asset = datasource.add_dataframe_asset(name='healthtrack_raw_asset')
batch_definition = asset.add_batch_definition_whole_dataframe('whole_df')
batch = batch_definition.get_batch(batch_parameters={'dataframe': df})

suite = context.suites.add(gx.ExpectationSuite(name='healthtrack_raw_suite'))
validator = context.get_validator(batch=batch, expectation_suite=suite)

# Every discharge must carry a patient identifier so care teams can route outreach to the right person.
validator.expect_column_values_to_not_be_null('patient_id')

# Every record must include known 30-day return status so historical outcomes can train and validate risk scoring.
validator.expect_column_values_to_not_be_null('readmission_30d')

# 30-day return status must be yes/no only to preserve clinical meaning in reports and models.
validator.expect_column_values_to_be_in_set('readmission_30d', [0, 1])

# Bill totals must stay in realistic encounter ranges so downstream dashboards are not distorted by error codes.
validator.expect_column_values_to_be_between('total_bill_usd', min_value=500, max_value=200000)

# Department labels must map to approved service lines for accurate routing and governance.
validator.expect_column_values_to_be_in_set('department', ['Cardiology', 'Oncology', 'Orthopedics', 'Neurology', 'General Medicine'])

validation_results = validator.validate()
validation_results['success']

In [ ]:
# Build Great Expectations Data Docs and save as healthtrack_gx_report.html
site_urls = context.build_data_docs()
local_site_url = site_urls.get('local_site')

if not local_site_url:
    raise RuntimeError('No local Data Docs site URL returned by Great Expectations.')

index_path = Path(local_site_url.replace('file://', ''))
if not index_path.exists():
    raise FileNotFoundError(f'Data Docs index file not found: {index_path}')

shutil.copyfile(index_path, 'healthtrack_gx_report.html')
print(f'Data Docs source: {index_path}')
print('Saved: healthtrack_gx_report.html')

Today's quality checkpoint shows that some discharge records are not yet safe for direct operational use: several are missing patient identity, several have no confirmed 30-day return outcome, and some include billing placeholder values that are not real charges. Until those records are corrected, daily care-priority lists and readmission performance reporting can miss patients who need follow-up and create avoidable coordination risk.